# Library Coverage Heatmap: 10-Minute Walk

This notebook draws an interactive map of Chicago that shows, for every census
tract, **what share of the tract lies within a 10-minute walk of an open
Chicago Public Library (CPL) branch**. Tracts are shaded from red (no coverage)
through yellow to green (fully covered), and every branch is marked with a blue
dot. A summary table at the end counts how many tracts and residents fall into
each coverage band.

### Where this fits

This notebook doesn't compute anything new. It **visualizes results the
coverage pipeline has already written** to the project's S3 bucket. Run the
`cpl-coverage` pipeline (the `cpl_coverage_pipeline.ipynb` notebook in the
*pipelines* workbench) at least once before opening this one. It reads three
pipeline outputs:

| File in S3 | Pipeline step that writes it | What it holds |
|---|---|---|
| `results/coverage_scores.parquet` | Step 4, *aggregate scores* | Coverage scores per tract, travel mode and travel time |
| `clean/population_demographics.csv` | Step 1b, *clean census tracts* | Tract outlines, names and population |
| `clean/branches.csv` | Step 1a, *clean branches* | Branch names, coordinates and open/closed status |

### Key terms

- **Census tract:** a small, stable area the U.S. Census Bureau uses to report
  statistics, typically home to 1,200–8,000 people. Chicago has more than 800.
- **Isochrone:** the area reachable from a point within a set time by a set
  mode of travel. The pipeline fetches a 10-minute walking isochrone for every
  branch.
- **Coverage:** how much of a tract's land area overlaps at least one branch's
  isochrone.
- **Choropleth:** a map where each area is filled with a color that represents
  a value. Here the value is coverage.

### Contents

1. Install the mapping libraries
2. Connect to S3
3. Load coverage scores
4. Load census tracts
5. Load library branches
6. Join scores onto tracts
7. Draw the interactive map
8. Summarize coverage by band

## 1. Install the mapping libraries

The workbench image ships the usual data science stack (`pandas`, `numpy`,
`boto3`, `matplotlib`) but not the mapping libraries this notebook needs:

- **`geopandas`**: extends pandas so a table can hold shapes (polygons, points)
  in a `geometry` column and understands map coordinate systems.
- **`folium`**: builds interactive web maps in Python. It generates a
  [Leaflet](https://leafletjs.com/) map that renders right in the notebook, with
  pan, zoom and hover.
- **`branca`**: folium's companion library for legends and color scales.

`!pip install` runs inside the workbench's Python environment. The packages
stay installed until the workbench restarts.

In [ ]:
!pip install -q folium branca geopandas

## 2. Connect to S3

All project data lives in an S3-compatible bucket provided by OpenShift Data
Foundation. The workbench has the bucket's connection details attached as a
*data connection* (the `cpl-s3` secret), which appears inside the notebook as
environment variables:

| Variable | Meaning |
|---|---|
| `AWS_S3_ENDPOINT` | Address of the S3 service inside the cluster |
| `AWS_S3_BUCKET` | Bucket name |
| `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` | Credentials for the bucket |

**`boto3`** is the standard Python client for S3. `verify=False` skips TLS
certificate checking, because the in-cluster endpoint uses a certificate issued
by the cluster's own certificate authority rather than a public one. The
traffic is still encrypted. It just isn't checked against a trusted list.

The cell also imports everything the rest of the notebook uses, including
**`shapely`** (the geometry engine underneath geopandas) and its `wkt` module
for reading shapes stored as text.

In [ ]:
import os
import io
import boto3
import pandas as pd
import geopandas as gpd
import folium
from shapely import wkt
from branca.colormap import LinearColormap

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    verify=False,
)
bucket = os.environ["AWS_S3_BUCKET"]

## 3. Load coverage scores

`results/coverage_scores.parquet` has one row per **tract × travel mode ×
travel time**. Parquet is a compact, typed, columnar file format that loads
much faster than CSV. The pipeline scores three combinations (driving 5 min,
cycling 10 min, walking 10 min), and this map focuses on walking because it
reflects access for people without a car. The filter keeps only
`mode == "walking"` and `travel_time == 10`.

| Column | Meaning |
|---|---|
| `geoid` | 11-digit census tract ID (state + county + tract), used to join to other tables |
| `percent_covered` | Share of the tract's area within reach of **at least one** branch, from 0 to 1. Overlapping branches are not double-counted. |
| `raw_score` | Sum of every branch's individual overlap. It can exceed 1 where branches overlap, so it rewards redundancy. |
| `num_branches` | Number of branches whose isochrone reaches the tract |

Tracts that no branch can reach have **no row at all** in this file. That
matters in step 6.

In [ ]:
obj = s3.get_object(Bucket=bucket, Key="results/coverage_scores.parquet")
scores = pd.read_parquet(io.BytesIO(obj["Body"].read()))
scores = scores[(scores["mode"] == "walking") & (scores["travel_time"] == 10)]
print(f"Coverage scores: {scores.shape}")
scores.head()

## 4. Load census tracts

`clean/population_demographics.csv` holds one row per tract inside the city,
with its outline and population. A CSV can't store shapes directly, so the
pipeline saved each outline as **WKT** (*well-known text*), a standard text
format such as `POLYGON ((-87.63 41.88, …))`.

The transformations:

1. `wkt.loads` turns each WKT string back into a shapely polygon.
2. `gpd.GeoDataFrame(..., crs="EPSG:4326")` makes the table spatially aware.
   **EPSG:4326** is the coordinate reference system for plain latitude and
   longitude, the same system GPS and web maps use.
3. Only the four columns the map needs are kept: `geoid`, `geometry`,
   `qualifying name` (a readable label such as *Census Tract 8391, Cook County,
   Illinois*) and `total population`.

In [ ]:
obj = s3.get_object(Bucket=bucket, Key="clean/population_demographics.csv")
tracts = pd.read_csv(io.BytesIO(obj["Body"].read()))
tracts["geometry"] = tracts["geometry"].apply(wkt.loads)
tracts = gpd.GeoDataFrame(tracts, geometry="geometry", crs="EPSG:4326")
tracts = tracts[["geoid", "geometry", "qualifying name", "total population"]]
print(f"Census tracts: {len(tracts)}")
tracts.head()

## 5. Load library branches

`clean/branches.csv` lists every branch with its latitude and longitude. The
table also includes a branch that has permanently closed (flagged
`PERMANENTLY_CLOSED = 1`), which is kept for historical analysis. The filter
removes it so the map shows only branches serving residents today, which are
also the only branches the coverage scores were computed for.

In [ ]:
obj = s3.get_object(Bucket=bucket, Key="clean/branches.csv")
branches = pd.read_csv(io.BytesIO(obj["Body"].read()))
branches = branches[branches["PERMANENTLY_CLOSED"] == 0].reset_index(drop=True)
print(f"Active branches: {len(branches)}")

## 6. Join scores onto tracts

This step attaches each tract's scores to its outline using the shared `geoid`
column.

- It's a **left join** starting from the tracts, so **every tract stays on the
  map**, including the ones no branch reaches. An inner join would silently
  drop exactly the tracts we most want to see.
- Those unreached tracts come out of the join with empty (`NaN`) scores, which
  are filled with 0. Zero is the true value: no coverage, no branches.
- `describe()` prints summary statistics for the three scores (count, mean,
  spread, quartiles). A quick read: if the median `percent_covered` is low,
  most tracts are only partly within walking distance of a library.

In [ ]:
merged = tracts.merge(
    scores[["geoid", "percent_covered", "raw_score", "num_branches"]],
    on="geoid",
    how="left",
)
merged["percent_covered"] = merged["percent_covered"].fillna(0)
merged["num_branches"] = merged["num_branches"].fillna(0).astype(int)
merged["raw_score"] = merged["raw_score"].fillna(0)
print(f"Merged tracts: {len(merged)}")
merged[["percent_covered", "raw_score", "num_branches"]].describe()

## 7. Draw the interactive map

How the map is built:

- **Base map:** `folium.Map` centers on downtown Chicago (41.8781, −87.6298).
  The muted *CartoDB Positron* tiles keep streets visible without competing
  with the coverage colors.
- **Color scale:** `LinearColormap` blends red (`#d73027`) → yellow (`#fee08b`)
  → green (`#1a9850`) across 0–1. That's a *diverging* scale, so the eye goes
  straight to both extremes: well-served areas and gaps.
- **Tracts:** `folium.GeoJson` draws every tract polygon. The `style_function`
  runs once per tract and picks its fill color from `percent_covered`; thin
  dark borders separate neighboring tracts.
- **Tooltip:** hover over a tract to see its name, coverage, number of branches
  in reach and population.
- **Branches:** each open branch is a blue `CircleMarker`. Blue isn't on the
  red–green scale, so branches never blend into the shading. Click one to see
  its name.
- **Legend:** `colormap.add_to(m)` adds the color bar in the corner.

**Reading the map:** green clusters around branches are expected. The useful
signal is red and orange tracts with large populations, which are candidate
areas for new or expanded library service. Two caveats:

- Coverage measures **land area**, not people. A tract that is half park or
  rail yard can score low even though most of its residents live near a
  branch.
- The isochrones follow the street network, so rivers, expressways and rail
  lines can leave a tract uncovered even when a branch is close in a straight
  line.

In [ ]:
m = folium.Map(location=[41.8781, -87.6298], zoom_start=11, tiles="cartodbpositron")

colormap = LinearColormap(
    colors=["#d73027", "#fee08b", "#1a9850"],
    vmin=0,
    vmax=1,
    caption="Percent Covered (walking, 10 min)",
)

folium.GeoJson(
    merged,
    style_function=lambda feature: {
        "fillColor": colormap(feature["properties"]["percent_covered"]),
        "color": "#333333",
        "weight": 0.5,
        "fillOpacity": 0.7,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["qualifying name", "percent_covered", "num_branches", "total population"],
        aliases=["Tract", "% Covered", "# Branches", "Population"],
        localize=True,
    ),
).add_to(m)

for _, branch in branches.iterrows():
    folium.CircleMarker(
        location=[branch["LATITUDE"], branch["LONGITUDE"]],
        radius=5,
        color="#2b83ba",
        fill=True,
        fill_color="#2b83ba",
        fill_opacity=0.9,
        popup=branch["BRANCH"],
    ).add_to(m)

colormap.add_to(m)
m

## 8. Summarize coverage by band

The map shows the pattern. This table puts numbers on it by sorting every tract
into one of four coverage bands and totaling the tracts and residents in each.

- `pd.cut` assigns each tract to a band using the edges −0.01, 0, 0.5, 0.9 and
  1.01. The tiny negative first edge creates a band holding **exactly 0%**, so
  completely unreached tracts are counted separately from barely-reached ones.
  The remaining bands are *above 0 up to 50%*, *above 50 up to 90%* and *above
  90%*.
- `groupby(..., observed=False)` keeps every band in the output, even one with
  no tracts.
- Population is formatted with thousands separators for readability.

**One thing to keep in mind:** a tract's whole population is counted in its
band. A tract that is 60% covered puts all of its residents in the 50–90% row,
even though some of them live in the uncovered part.

In [ ]:
bins = pd.cut(
    merged["percent_covered"],
    bins=[-0.01, 0.0, 0.50, 0.90, 1.01],
    labels=["0%", "1-50%", "50-90%", "90%+"],
)
summary = merged.groupby(bins, observed=False).agg(
    tracts=("geoid", "count"),
    total_population=("total population", "sum"),
).reset_index()
summary.columns = ["Coverage Bucket", "Tracts", "Total Population"]
summary["Total Population"] = summary["Total Population"].apply(lambda x: f"{x:,.0f}")
summary